In [88]:
# !pip install requests
# !pip install beautifulsoup4
# !pip install langchain-community
# !pip install langchain-text-splitters
# !pip install langchain-chroma
# !pip install -U langchain-ollama
# !pip install -U pypdf

In [182]:
import os

# ตั้งค่าให้ Ollama ใช้เซิร์ฟเวอร์ที่รันอยู่บนเครื่องอื่น
os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"

# ตรวจสอบว่าตัวแปรถูกตั้งค่าแล้วหรือไม่
print(os.getenv("USER_AGENT"))

Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36


In [183]:
import bs4
import requests
from langchain_community.document_loaders import WebBaseLoader

# Monkey Patch requests.get เพื่อใส่ User-Agent
original_get = requests.get

def patched_get(url, *args, **kwargs):
    headers = kwargs.pop("headers", {})
    headers["User-Agent"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
    return original_get(url, headers=headers, *args, **kwargs)

requests.get = patched_get  # ใช้ patched version ของ requests.get

# ใช้ WebBaseLoader ตามปกติ
loader = WebBaseLoader(
    web_paths=("http://ollama:11434",),
)
docs = loader.load()

# คืนค่า requests.get กลับเป็นปกติ (ถ้าต้องการ)
requests.get = original_get

print(docs)

[Document(metadata={'source': 'http://ollama:11434'}, page_content='Ollama is running')]


In [184]:
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain_ollama import OllamaEmbeddings
from langchain_ollama.llms import OllamaLLM
import sys

In [185]:
import os

# ตั้งค่าให้ Ollama ใช้เซิร์ฟเวอร์ที่รันอยู่บนเครื่องอื่น
os.environ["OLLAMA_BASE_URL"] = "http://ollama:11434"

# ตรวจสอบว่าตัวแปรถูกตั้งค่าแล้วหรือไม่
print(os.getenv("OLLAMA_BASE_URL"))  # ควรแสดง http://server-ip:11434

http://ollama:11434


In [186]:
OLLAMA_BASE_URL = "http://ollama:11434"

In [187]:
import chromadb
chromadb.api.client.SharedSystemClient.clear_system_cache()

In [188]:
import os
import shutil
import time

def ingest(file_path: str):
    # Delete the folder if it exists
    db_path = "./sql_chroma_db"
    if os.path.exists(db_path):
        shutil.rmtree(db_path)
        print(f"Deleted existing folder: {db_path}")

    # Add a 1-second delay before loading the PDF
    time.sleep(3)
        
    # Get the doc
    loader = PyPDFLoader(file_path)
    pages = loader.load_and_split()
    # Split the pages by char
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1024,
        chunk_overlap=100,
        length_function=len,
        add_start_index=True,
    )
    chunks = text_splitter.split_documents(pages)
    print(f"Split {len(pages)} documents into {len(chunks)} chunks.")
    
    
    local_embeddings = OllamaEmbeddings(
        model="all-minilm",
        base_url=OLLAMA_BASE_URL
    )
    
    # Create vector store
    Chroma.from_documents(documents=chunks,  embedding=local_embeddings, persist_directory=db_path)

In [190]:
# only run this once to generate vector store
# ingest("sql_tutorial.pdf")

ingest("sql_tutorial.pdf")

Split 33 documents into 76 chunks.


In [173]:
# Create a RAG chain that retreives relevent chunks and prepares a response

In [191]:
def rag_chain():
    model = OllamaLLM(
        model="deepseek-r1:8b",
        base_url=OLLAMA_BASE_URL
    )
    

    
    #Load vector store
    embedding = OllamaEmbeddings(
        model="all-minilm",
        base_url=OLLAMA_BASE_URL
    )
    vector_store = Chroma(persist_directory="./sql_chroma_db", embedding_function=embedding)

    #Create chain
    retriever = vector_store.as_retriever(
        search_type="similarity",
        # search_type="similarity_score_threshold",
        search_kwargs={
            "k": 3,
            # "score_threshold": 0.7, # คืนค่าเอกสารที่มีคะแนนความคล้ายคลึง ≥ 0.7
            # "fetch_k": 10,  # ดึงผลลัพธ์มา 10 รายการก่อนเลือก 3 รายการที่หลากหลายที่สุด
            # "lambda_mult": 0.5,  # ปรับสมดุลระหว่างความคล้ายคลึงและความหลากหลาย
        },
    )
    
    # # Get documents from the vector store
    # results = retriever.invoke("ถ้าต้องถูกทำงานล่วงเวลาในวันหยุด เราจะได้ค่าจ้างเท่าไหร?")
    
    # # Print the results to see what documents are returned
    # print("Vector Store Results:")
    # for idx, result in enumerate(results):
    #     print(f"Result {idx+1}: {result}")

    prompt = PromptTemplate.from_template(
        """
        <s> [Instructions] You are a friendly assistant. Answer the question based only on the following context. 
        If you don't know the answer, then reply, No Context availabel for this question {input}. [/Instructions] </s> 
        [Instructions] Question: {input}
        Context: {context} 
        Answer: [/Instructions]
        """
    )

    document_chain = create_stuff_documents_chain(model, prompt)
    chain = create_retrieval_chain(retriever, document_chain)
    
    return chain, prompt


In [192]:
def ask(query: str):
    # Initialize chain and prompt
    chain, prompt = rag_chain()

    # Display the prompt before invoking
    print("Prompt used in the chain:")
    print(prompt.format(input=query, context=""))

    # Invoke chain with the query
    result = chain.invoke({"input": query})

    # Print the result
    print("Answer: ", result["answer"])
    
    # Show the source documents
    for doc in result["context"]:
        print("Source: ", doc.metadata["source"])

        
    # #
    # chain = rag_chain()

    # # Display the prompt before invoking
    # prompt = chain.prompts[0].template  # Assuming the first prompt is the one used
    # print("Prompt used in the chain:")
    # print(prompt.format(input=query, context=""))
    
    # # invoke chain
    # result = chain.invoke({"input": query})
    
    # print(result["answer"])
    
    # for doc in result["context"]:
    #     print("Source: ", doc.metadata["source"])

In [ ]:
ask("ถ้าต้องถูกทำงานล่วงเวลาในวันหยุด เราจะได้ค่าจ้างเท่าไหร?")

Prompt used in the chain:

        <s> [Instructions] You are a friendly assistant. Answer the question based only on the following context. 
        If you don't know the answer, then reply, No Context availabel for this question ถ้าต้องถูกทำงานล่วงเวลาในวันหยุด เราจะได้ค่าจ้างเท่าไหร?. [/Instructions] </s> 
        [Instructions] Question: ถ้าต้องถูกทำงานล่วงเวลาในวันหยุด เราจะได้ค่าจ้างเท่าไหร?
        Context:  
        Answer: [/Instructions]
        


In [ ]:
ask("ไม่สามารถรับเด็กอายุต่ำกว่ากี่ปีให้มาทำงานได้")

Prompt used in the chain:

        <s> [Instructions] You are a friendly assistant. Answer the question based only on the following context. 
        If you don't know the answer, then reply, No Context availabel for this question ไม่สามารถรับเด็กอายุต่ำกว่ากี่ปีให้มาทำงานได้. [/Instructions] </s> 
        [Instructions] Question: ไม่สามารถรับเด็กอายุต่ำกว่ากี่ปีให้มาทำงานได้
        Context:  
        Answer: [/Instructions]
        
